In [1]:
import os
import re
import json
import base64
from io import StringIO
from typing import List, Dict, Any, Optional, TypedDict
from collections import Counter

from unstructured.staging.base import elements_to_json, elements_from_json
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END

import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output

import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import camelot
import pikepdf
import chromadb

load_dotenv()
plt.rcParams['font.family'] = 'DejaVu Sans'

os.makedirs("extracted_data/images", exist_ok=True)
os.makedirs("extracted_data/tables", exist_ok=True)
os.makedirs("document_store", exist_ok=True)


In [2]:
def remove_pdf_restrictions(input_path, output_path=None):
    """Strip extraction-restriction flags from a PDF (some PDFs disable text extraction)"""
    if output_path is None:
        output_path = input_path.replace(".pdf", "_unlocked.pdf")
    with pikepdf.open(input_path, allow_overwriting_input=True) as pdf:
        pdf.save(output_path)
    print(f"✅ Saved unrestricted copy to {output_path}")
    return output_path

In [3]:
def partition_document(file_path: str, store_dir: str = "document_store"):
    """Extract elements from PDF, storing the parsed JSON on disk"""
    os.makedirs(store_dir, exist_ok=True)

    pdf_name = os.path.splitext(os.path.basename(file_path))[0]
    store_path = os.path.join(store_dir, f"{pdf_name}_elements.json")

    if os.path.exists(store_path):
        print(f"✅ Found stored elements at {store_path} — loading from disk")
        elements = elements_from_json(store_path)
        print(f"✅ Loaded {len(elements)} elements from disk")
        return elements

    print(f"📄 Partitioning document: {file_path} (this may take a while for hi_res)")
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )
    print(f"✅ Extracted {len(elements)} elements")

    elements_to_json(elements, filename=store_path)
    print(f"💾 Stored elements to {store_path}")
    return elements


file_path = "docs/embedded-images-tables.pdf"
elements = partition_document(file_path)


✅ Found stored elements at document_store\embedded-images-tables_elements.json — loading from disk
✅ Loaded 12 elements from disk


In [4]:
def find_tables_camelot(pdf_path, min_accuracy=70, min_columns=2):
    """Run Camelot as a second opinion on table detection"""
    all_tables = {}
    for flavor in ["stream", "lattice"]:
        try:
            results = camelot.read_pdf(pdf_path, pages="all", flavor=flavor)
            for t in results:
                acc = t.parsing_report.get("accuracy", 0)
                if acc >= min_accuracy and t.df.shape[1] >= min_columns:
                    all_tables.setdefault(t.page, []).append({
                        "html": t.df.to_html(index=False, header=False),
                        "accuracy": acc
                    })
        except Exception as e:
            print(f"⚠️ Camelot {flavor} failed: {e}")
    return all_tables


def score_table_confidence(html, page_text_near_table=""):
    """Heuristically score whether a Camelot-detected 'table' is real or a false positive"""
    soup = BeautifulSoup(html, 'html.parser')
    cells = [c.get_text(strip=True) for c in soup.find_all(['td', 'th'])]
    cells = [c for c in cells if c]
    if not cells:
        return 0, ["no cell content"]

    score = 0
    reasons = []

    if re.search(r'\btable\s+\d+\b', page_text_near_table, re.IGNORECASE):
        score += 40
        reasons.append("+40: 'Table N' caption found nearby")

    numeric_ratio = sum(1 for c in cells if re.search(r'\d', c)) / len(cells)
    if numeric_ratio > 0.3:
        score += 25
        reasons.append(f"+25: numeric ratio {numeric_ratio:.0%}")
    elif numeric_ratio < 0.05:
        score -= 15
        reasons.append(f"-15: almost no numbers ({numeric_ratio:.0%})")

    avg_len = sum(len(c) for c in cells) / len(cells)
    if avg_len < 25:
        score += 20
        reasons.append(f"+20: short cells (avg {avg_len:.0f} chars)")
    elif avg_len > 60:
        score -= 25
        reasons.append(f"-25: long cells, likely prose (avg {avg_len:.0f} chars)")

    word_counts = Counter(cells)
    most_common_count = word_counts.most_common(1)[0][1] if word_counts else 0
    repetition_ratio = most_common_count / len(cells)
    if repetition_ratio > 0.15:
        score -= 30
        reasons.append(f"-30: high repetition ({repetition_ratio:.0%}) — likely figure/diagram text")

    return max(0, min(100, score)), reasons


def filter_camelot_tables(camelot_tables, elements, min_score=60):
    """Auto-classify each Camelot table as real or false-positive"""
    confirmed = {}
    page_text = {}
    for el in elements:
        pg = getattr(el.metadata, 'page_number', None)
        if pg:
            page_text.setdefault(pg, "")
            page_text[pg] += " " + el.text

    for page, tables_on_page in camelot_tables.items():
        for t in tables_on_page:
            score, reasons = score_table_confidence(t['html'], page_text.get(page, ""))
            if score >= min_score:
                confirmed.setdefault(page, []).append(t)

    print(f"✅ Confirmed real tables on pages: {sorted(confirmed.keys())}")
    return confirmed


camelot_tables_raw = find_tables_camelot(file_path)
confirmed_tables = filter_camelot_tables(camelot_tables_raw, elements)

# Only keep pages unstructured actually missed
unstructured_table_pages = {
    el.metadata.page_number for el in elements
    if type(el).__name__ == "Table" and hasattr(el.metadata, "page_number")
}
recovered_tables = {
    page: [t["html"] for t in tables]
    for page, tables in confirmed_tables.items()
    if page not in unstructured_table_pages
}
print(f"📋 Tables to inject (missed by unstructured): {list(recovered_tables.keys())}")

✅ Confirmed real tables on pages: [1]
📋 Tables to inject (missed by unstructured): []


In [5]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    print(f"✅ Created {len(chunks)} chunks")
    return chunks


chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 3 chunks


In [6]:
def crop_table_from_pdf(pdf_path, page_number, element, output_path, zoom=3, padding=8):
    """Crop the exact table region from the original PDF page — pixel-perfect"""
    try:
        coords = element.metadata.coordinates
        if not coords or not coords.points:
            return None

        doc = fitz.open(pdf_path)
        page = doc[page_number - 1]
        page_rect = page.rect  # actual PDF page size, in points

        # unstructured's coordinates may be in pixel space (hi_res rendering), not
        # PDF point space — scale to match the real page dimensions
        coord_system = coords.system
        system_width = getattr(coord_system, 'width', None)
        system_height = getattr(coord_system, 'height', None)

        if system_width and system_height:
            scale_x = page_rect.width / system_width
            scale_y = page_rect.height / system_height
        else:
            scale_x = scale_y = 1.0

        xs = [p[0] * scale_x for p in coords.points]
        ys = [p[1] * scale_y for p in coords.points]
        x0, x1 = min(xs) - padding, max(xs) + padding
        y0, y1 = min(ys) - padding, max(ys) + padding

        if x1 - x0 < 5 or y1 - y0 < 5:
            doc.close()
            return None

        x0, y0 = max(x0, 0), max(y0, 0)
        x1, y1 = min(x1, page_rect.width), min(y1, page_rect.height)

        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, clip=fitz.Rect(x0, y0, x1, y1))
        pix.save(output_path)
        doc.close()
        return output_path

    except Exception as e:
        print(f"     ⚠️ Could not crop table from PDF: {e}")
        return None


def clean_cell_text(cell):
    """Extract cell text, converting <sup>/<sub> tags to unicode"""
    sup_map = {'2': '²', '3': '³', '1': '¹'}
    text = ''
    for content in cell.contents:
        if getattr(content, 'name', None) == 'sup':
            raw = content.get_text()
            text += sup_map.get(raw, f'^{raw}')
        elif getattr(content, 'name', None) == 'sub':
            text += f"_{content.get_text()}"
        else:
            text += str(content) if isinstance(content, str) else content.get_text()
    return text.strip()


def html_table_to_image(html, output_path, title=None):
    """Fallback: render table from HTML if PDF cropping isn't available"""
    try:
        soup = BeautifulSoup(html, 'html.parser')
        rows = soup.find_all('tr')
        if not rows:
            return None
        data = [[clean_cell_text(c) for c in row.find_all(['td', 'th'])] for row in rows]
        header, body = data[0], data[1:]
        if not body or not header:
            return None
        col_count = len(header)
        body = [row + [''] * (col_count - len(row)) if len(row) < col_count else row[:col_count] for row in body]
        df = pd.DataFrame(body, columns=header)
    except Exception as e:
        print(f"     ⚠️ Could not parse table HTML: {e}")
        return None

    if df.empty:
        return None

    fig, ax = plt.subplots(figsize=(max(6, len(df.columns) * 1.4), max(1.5, len(df) * 0.5 + 1)))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    tbl = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#4a4a4a')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#f5f5f5')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    return output_path

In [7]:
def unique_path(path):
    """If file already exists, append _1, _2, etc. to avoid overwriting"""
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    counter = 1
    while os.path.exists(f"{base}_{counter}{ext}"):
        counter += 1
    return f"{base}_{counter}{ext}"


def separate_content_types(chunk, chunk_id, pdf_path, recovered_tables):
    """Analyze content types AND persist images/tables to disk with paths"""
    content_data = {'text': chunk.text, 'tables': [], 'images': [], 'types': ['text'], 'page': None}

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            if content_data['page'] is None and hasattr(element.metadata, 'page_number'):
                content_data['page'] = element.metadata.page_number

            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                table_idx = len(content_data['tables'])
                table_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.html")
                with open(table_path, 'w', encoding='utf-8') as f:
                    f.write(f"<html><body>{table_html}</body></html>")

                img_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.png")
                rendered = crop_table_from_pdf(pdf_path, content_data['page'], element, img_path)
                if rendered is None:
                    rendered = html_table_to_image(table_html, img_path, title=f"Table (page {content_data['page']})")

                content_data['tables'].append({"html": table_html, "path": table_path, "image_path": rendered})

            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    img_b64 = element.metadata.image_base64
                    img_idx = len(content_data['images'])
                    img_out_path = unique_path(f"extracted_data/images/chunk_{chunk_id}_img_{img_idx}.png")
                    with open(img_out_path, 'wb') as f:
                        f.write(base64.b64decode(img_b64))
                    content_data['images'].append({"base64": img_b64, "path": img_out_path})

        # Inject Camelot-recovered tables for this chunk's page (unstructured missed these)
        if content_data['page'] in recovered_tables:
            for html in recovered_tables[content_data['page']]:
                content_data['types'].append('table')
                table_idx = len(content_data['tables'])
                table_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.html")
                with open(table_path, 'w', encoding='utf-8') as f:
                    f.write(f"<html><body>{html}</body></html>")
                img_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.png")
                rendered = html_table_to_image(html, img_path, title=f"Table (page {content_data['page']}, recovered)")
                content_data['tables'].append({"html": html, "path": table_path, "image_path": rendered})
            del recovered_tables[content_data['page']]

    content_data['types'] = list(set(content_data['types']))
    return content_data


In [8]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    try:
        model_name = "qwen/qwen3.6-27b" if images else "openai/gpt-oss-120b"
        llm = ChatGroq(model_name=model_name, temperature=0)

        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}
        """

        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
            prompt_text += """
IMPORTANT: Scan the TEXT CONTENT above for any explicit table number, label, or caption
(e.g. "Table 3"). If found, state it verbatim as the FIRST LINE of your description in
the format: "This is Table X: <topic>". Always include the exact table number if present.
"""

        prompt_text += """
        YOUR TASK:
        Generate a comprehensive, searchable description covering key facts, main topics,
        questions this content could answer, visual content analysis, and alternative search terms.

        SEARCHABLE DESCRIPTION:"""

        if images:
            message_content = [{"type": "text", "text": prompt_text}]
            for image_base64 in images:
                message_content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}})
        else:
            message_content = prompt_text

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content

    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [9]:
def save_processed_chunks(documents, store_path):
    """Save processed_chunks (LangChain Documents) to disk as JSON"""
    data = [
        {"page_content": doc.page_content, "metadata": doc.metadata}
        for doc in documents
    ]
    with open(store_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved {len(documents)} processed chunks to {store_path}")


def load_processed_chunks(store_path):
    """Load processed_chunks back into LangChain Documents"""
    with open(store_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    documents = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in data]
    print(f"✅ Loaded {len(documents)} processed chunks from disk")
    return documents


def summarise_chunks(chunks, pdf_path, recovered_tables, store_dir="document_store"):
    """Process all chunks with AI Summaries, saving images/tables to disk.
    Uses a stored JSON if available, to avoid re-running Groq summarization."""

    os.makedirs(store_dir, exist_ok=True)
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    store_path = os.path.join(store_dir, f"{pdf_name}_processed_chunks.json")

    if os.path.exists(store_path):
        print(f"✅ Found stored processed chunks at {store_path} — skipping re-summarization")
        return load_processed_chunks(store_path)

    print("🧠 Processing chunks with AI Summaries...")
    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        print(f"   Processing chunk {i+1}/{total_chunks}")
        content_data = separate_content_types(chunk, chunk_id=i, pdf_path=pdf_path, recovered_tables=recovered_tables)
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        table_htmls = [t["html"] for t in content_data['tables']]
        image_b64s = [img["base64"] for img in content_data['images']]

        if table_htmls or image_b64s:
            enhanced_content = create_ai_enhanced_summary(content_data['text'], table_htmls, image_b64s)
        else:
            enhanced_content = content_data['text']

        doc = Document(
            page_content=enhanced_content,
            metadata={
                "chunk_id": i,
                "page": content_data['page'] or 0,
                "raw_text": content_data['text'],  # Preserved in full for generation & reranking
                "table_paths": json.dumps([t["path"] for t in content_data['tables']]),
                "table_image_paths": json.dumps([t["image_path"] for t in content_data['tables'] if t.get("image_path")]),
                "image_paths": json.dumps([img["path"] for img in content_data['images']]),
                "has_table": bool(content_data['tables']),
                "has_image": bool(content_data['images']),
            }
        )
        langchain_documents.append(doc)

    print(f"✅ Processed {len(langchain_documents)} chunks")

    save_processed_chunks(langchain_documents, store_path)
    return langchain_documents


processed_chunks = summarise_chunks(chunks, pdf_path=file_path, recovered_tables=recovered_tables)


✅ Found stored processed chunks at document_store\embedded-images-tables_processed_chunks.json — skipping re-summarization
✅ Loaded 3 processed chunks from disk


In [10]:
class NomicEmbeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        return super().embed_documents([f"search_document: {t}" for t in texts])
    def embed_query(self, text):
        return super().embed_query(f"search_query: {text}")


class MultiNamespaceVectorStore:
    """A single ChromaDB with per-PDF namespace isolation via separate collections.
    Exposes similarity_search() that queries across ALL namespaces."""

    def __init__(self, persist_directory="vector_store/chroma_db"):
        self.persist_directory = persist_directory
        self.embedding_model = NomicEmbeddings(
            model_name="nomic-ai/nomic-embed-text-v1.5",
            model_kwargs={"trust_remote_code": True}
        )
        self._namespaces = {}

    def add_pdf(self, documents, pdf_name):
        """Add a PDF's chunks as a separate namespace (collection). Skips if already indexed."""
        collection_name = re.sub(r'[^a-zA-Z0-9_-]', '_', pdf_name)[:63]

        existing = Chroma(
            collection_name=collection_name,
            persist_directory=self.persist_directory,
            embedding_function=self.embedding_model,
            collection_metadata={"hnsw:space": "cosine"}
        )
        if existing._collection.count() > 0:
            print(f"✅ Namespace '{pdf_name}' already indexed ({existing._collection.count()} chunks) — skipping")
            self._namespaces[pdf_name] = existing
            return

        for doc in documents:
            doc.metadata["source_pdf"] = pdf_name

        doc_ids = [f"{collection_name}_chunk_{i}" for i in range(len(documents))]

        print(f"🔮 Indexing '{pdf_name}' into namespace ({len(documents)} chunks)...")
        vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embedding_model,
            persist_directory=self.persist_directory,
            collection_name=collection_name,
            collection_metadata={"hnsw:space": "cosine"},
            ids=doc_ids
        )
        print(f"✅ Namespace '{pdf_name}' indexed")
        self._namespaces[pdf_name] = vectorstore

    def similarity_search(self, query, k=15):
        """Search across ALL namespaces, return top-k combined results."""
        all_results = []
        per_ns_k = max(k, 10)

        for name, store in self._namespaces.items():
            try:
                results = store.similarity_search_with_relevance_scores(query, k=per_ns_k)
                all_results.extend(results)
            except Exception as e:
                print(f"⚠️ Search failed in namespace '{name}': {e}")

        all_results.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, score in all_results[:k]]

    def delete_namespace(self, pdf_name):
        """Remove a PDF's entire namespace"""
        
        collection_name = re.sub(r'[^a-zA-Z0-9_-]', '_', pdf_name)[:63]
        client = chromadb.PersistentClient(path=self.persist_directory)
        try:
            client.delete_collection(collection_name)
            self._namespaces.pop(pdf_name, None)
            print(f"🗑️ Deleted namespace '{pdf_name}'")
        except Exception as e:
            print(f"⚠️ Could not delete namespace '{pdf_name}': {e}")

    def list_namespaces(self):
        """List all indexed PDFs"""
    
        client = chromadb.PersistentClient(path=self.persist_directory)
        collections = client.list_collections()
        for c in collections:
            print(f"  📄 {c.name} ({c.count()} chunks)")
        return collections


# --- Process all PDFs and build the shared vector store ---
pdf_files = [
    "docs/attention-is-all-you-need.pdf",
    # Add more PDFs here:
    "docs/somatosensory.pdf",
    "docs/embedded-images-tables.pdf"
]

db = MultiNamespaceVectorStore()
all_processed_chunks = []

for file_path in pdf_files:
    elements = partition_document(file_path)

    camelot_tables_raw = find_tables_camelot(file_path)
    confirmed_tables = filter_camelot_tables(camelot_tables_raw, elements)
    unstructured_table_pages = {
        el.metadata.page_number for el in elements
        if type(el).__name__ == "Table" and hasattr(el.metadata, "page_number")
    }
    recovered_tables = {
        page: [t["html"] for t in tables]
        for page, tables in confirmed_tables.items()
        if page not in unstructured_table_pages
    }

    chunks = create_chunks_by_title(elements)
    processed = summarise_chunks(chunks, pdf_path=file_path, recovered_tables=recovered_tables)

    # Tag each chunk with source_pdf
    pdf_name = os.path.splitext(os.path.basename(file_path))[0]
    for doc in processed:
        doc.metadata["source_pdf"] = pdf_name

    all_processed_chunks.extend(processed)
    db.add_pdf(processed, pdf_name)

print(f"\n📊 Total chunks across all PDFs: {len(all_processed_chunks)}")


<All keys matched successfully>


✅ Found stored elements at document_store\attention-is-all-you-need_elements.json — loading from disk
✅ Loaded 266 elements from disk


c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (98.0, 94.5072576, 514.3140640576001, 234.60110706666666)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (98.0, 659.6790784, 513.9972105879998, 768.325786877612)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


✅ Confirmed real tables on pages: [8, 9, 10]
🔨 Creating smart chunks...
✅ Created 33 chunks
✅ Found stored processed chunks at document_store\attention-is-all-you-need_processed_chunks.json — skipping re-summarization
✅ Loaded 33 processed chunks from disk
✅ Namespace 'attention-is-all-you-need' already indexed (33 chunks) — skipping
✅ Found stored elements at document_store\somatosensory_elements.json — loading from disk
✅ Loaded 59 elements from disk


c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (46.6929, 320.55449999999996, 552.2591, 666.8203667714286)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


✅ Confirmed real tables on pages: []
🔨 Creating smart chunks...
✅ Created 9 chunks
✅ Found stored processed chunks at document_store\somatosensory_processed_chunks.json — skipping re-summarization
✅ Loaded 9 processed chunks from disk
✅ Namespace 'somatosensory' already indexed (9 chunks) — skipping
✅ Found stored elements at document_store\embedded-images-tables_elements.json — loading from disk
✅ Loaded 12 elements from disk
✅ Confirmed real tables on pages: [1]
🔨 Creating smart chunks...
✅ Created 3 chunks
✅ Found stored processed chunks at document_store\embedded-images-tables_processed_chunks.json — skipping re-summarization
✅ Loaded 3 processed chunks from disk
✅ Namespace 'embedded-images-tables' already indexed (3 chunks) — skipping

📊 Total chunks across all PDFs: 45


In [11]:
def tokenize_text(text: str) -> List[str]:
    """Robust tokenization for BM25: lowercase, strip punctuation, preserve alphanumeric terms."""
    if not text:
        return []
    return re.findall(r'[a-zA-Z0-9_-]+', text.lower())


def build_bm25_index(processed_chunks):
    """Build a keyword-search index alongside the vector store with clean tokenization"""
    tokenized = [
        tokenize_text(doc.page_content + " " + doc.metadata.get("raw_text", ""))
        for doc in processed_chunks
    ]
    bm25 = BM25Okapi(tokenized)
    print(f"✅ BM25 index built over {len(processed_chunks)} chunks")
    return bm25


bm25_index = build_bm25_index(all_processed_chunks)


✅ BM25 index built over 45 chunks


In [12]:
print("🔄 Loading reranker model...")
reranker_model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
try:
    reranker = CrossEncoder(reranker_model_name)
    print(f"✅ Reranker loaded: {reranker_model_name}")
except Exception as e:
    print(f"⚠️ Error loading {reranker_model_name}: {e}")
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


🔄 Loading reranker model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [13]:
def hybrid_search(query: str, db, bm25_index, processed_chunks, k=15, alpha=0.5) -> List[Document]:
    """Combine semantic (Chroma) and keyword (BM25) search via Reciprocal Rank Fusion.
    alpha: weight toward semantic (1.0) vs keyword (0.0); 0.5 = balanced"""

    semantic_results = db.similarity_search(query, k=k)
    query_tokens = tokenize_text(query)
    if query_tokens:
        bm25_scores = bm25_index.get_scores(query_tokens)
        bm25_ranked_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
    else:
        bm25_ranked_idx = []

    rrf_scores = {}
    for rank, doc in enumerate(semantic_results):
        cid = (doc.metadata.get("source_pdf", ""), doc.metadata.get("chunk_id"))
        rrf_scores[cid] = rrf_scores.get(cid, 0) + alpha * (1.0 / (rank + 60))

    for rank, idx in enumerate(bm25_ranked_idx):
        cid = (processed_chunks[idx].metadata.get("source_pdf", ""), processed_chunks[idx].metadata.get("chunk_id"))
        rrf_scores[cid] = rrf_scores.get(cid, 0) + (1.0 - alpha) * (1.0 / (rank + 60))

    top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]
    id_to_doc = {(d.metadata.get("source_pdf", ""), d.metadata.get("chunk_id")): d for d in processed_chunks}
    return [id_to_doc[i] for i in top_ids if i in id_to_doc]


def rerank(query: str, candidates: List[Document], top_k=5) -> List[Document]:
    """Cross-encoder reranking evaluating both AI summaries and raw text context."""
    if not candidates:
        return []

    pairs = []
    for doc in candidates:
        meta = doc.metadata or {}
        raw_text = meta.get("raw_text", "")
        summary = doc.page_content if doc.page_content != raw_text else ""
        combined_text = f"{summary}\n{raw_text}".strip() if summary else raw_text
        pairs.append((query, combined_text[:2500]))

    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked[:top_k]]


In [14]:
# =============================================================================
# LangGraph State & Node Definitions for Multi-Modal RAG
# =============================================================================

class RAGState(TypedDict):
    query: str
    transformed_query: str
    retrieved_docs: List[Document]
    reranked_docs: List[Document]
    filtered_docs: List[Document]
    visual_elements: List[Dict[str, Any]]
    answer: str
    sources: List[Dict[str, Any]]
    retry_count: int


def clean_model_output(output_text: str) -> str:
    """Remove reasoning traces like <think>...</think> if returned by reasoning models."""
    cleaned = re.sub(r'<think>.*?</think>', '', output_text, flags=re.DOTALL).strip()
    return cleaned if cleaned else output_text.strip()


# Node 1: Query Normalization & Preprocessing
def transform_query_node(state: RAGState) -> Dict[str, Any]:
    query = state["query"].strip()
    # Clean extra whitespace and format query for retrieval
    transformed = re.sub(r'\s+', ' ', query)
    return {"transformed_query": transformed}


# Node 2: Hybrid Retrieval (Chroma Dense + BM25 Sparse + RRF)
def retrieve_node(state: RAGState) -> Dict[str, Any]:
    query = state.get("transformed_query", state["query"])
    candidates = hybrid_search(query, db, bm25_index, all_processed_chunks, k=15)
    return {"retrieved_docs": candidates}


# Node 3: Cross-Encoder Reranking
def rerank_node(state: RAGState) -> Dict[str, Any]:
    query = state.get("transformed_query", state["query"])
    candidates = state.get("retrieved_docs", [])
    ranked_docs = rerank(query, candidates, top_k=5)
    return {"reranked_docs": ranked_docs}


# Node 4: Grade Documents / Quality Filter (Corrective RAG)
def grade_documents_node(state: RAGState) -> Dict[str, Any]:
    docs = state.get("reranked_docs", [])
    # Ensure we have valid candidate documents
    valid_docs = [d for d in docs if (d.page_content or d.metadata.get("raw_text"))]
    current_retries = state.get("retry_count", 0)
    return {"filtered_docs": valid_docs, "retry_count": current_retries + 1}


# Node 5: Prepare Multi-Modal Context (Text + HTML Tables + Rendered Images)
def prepare_multimodal_context_node(state: RAGState) -> Dict[str, Any]:
    docs = state.get("filtered_docs", [])
    visual_elements = []
    sources = []

    for i, chunk in enumerate(docs):
        meta = chunk.metadata or {}
        source_pdf = meta.get("source_pdf", "Unknown PDF")
        page_num = meta.get("page", "Unknown")
        
        table_paths = json.loads(meta.get("table_paths", "[]"))
        table_image_paths = json.loads(meta.get("table_image_paths", "[]"))
        image_paths = json.loads(meta.get("image_paths", "[]"))

        # Track visual elements
        for img_idx, p in enumerate(image_paths):
            if os.path.exists(p):
                visual_elements.append({"label": f"Doc {i+1} (p.{page_num}) - Fig {img_idx+1}", "path": p})
        for tbl_img_idx, p in enumerate(table_image_paths):
            if os.path.exists(p):
                visual_elements.append({"label": f"Doc {i+1} (p.{page_num}) - Table {tbl_img_idx+1}", "path": p})

        source_entry = {
            "chunk_id": meta.get("chunk_id"),
            "source_pdf": source_pdf,
            "page": page_num,
            "type": "text",
            "preview": chunk.page_content[:180],
            "paths": []
        }
        if table_paths:
            source_entry["type"] = "table"
            source_entry["paths"].extend(table_image_paths if table_image_paths else table_paths)
        if image_paths:
            source_entry["type"] = "image" if not table_paths else "table+image"
            source_entry["paths"].extend(image_paths)

        sources.append(source_entry)

    return {"visual_elements": visual_elements, "sources": sources}


# Node 6: Multi-Modal VLM Generation
def generate_node(state: RAGState) -> Dict[str, Any]:
    docs = state.get("filtered_docs", [])
    query = state["query"]
    visual_elements = state.get("visual_elements", [])
    sources = state.get("sources", [])

    system_prompt = (
        "You are an expert multi-modal research assistant. Answer the user's question accurately "
        "and comprehensively using ONLY the provided document excerpts, tables, and images.\n\n"
        "Guidelines:\n"
        "1. Ground every claim in the provided context. Cite the specific Document number and Page number.\n"
        "2. If tables or images are provided, analyze their numerical data and visual contents directly.\n"
        "3. If the user provides a quote or sentence fragment from the text, explain what it refers to "
        "   and provide the full relevant context from the document.\n"
        "4. If the context does not contain enough information to answer, state: "
        "   'I do not have enough information in the provided documents to answer that question.'\n"
        "5. Provide a direct, well-structured response without leaking chain-of-thought or reasoning tags."
    )

    context_text = "CONTEXT DOCUMENTS:\n\n"
    for i, chunk in enumerate(docs):
        meta = chunk.metadata or {}
        source_pdf = meta.get("source_pdf", "Unknown PDF")
        page_num = meta.get("page", "Unknown")
        raw_text = meta.get("raw_text", chunk.page_content)
        table_paths = json.loads(meta.get("table_paths", "[]"))

        context_text += f"=== Document {i+1} [Source: {source_pdf}, Page {page_num}] ===\n"
        context_text += f"TEXT:\n{raw_text}\n\n"
        if table_paths:
            context_text += "TABLE DATA:\n"
            for t_idx, p in enumerate(table_paths):
                if os.path.exists(p):
                    with open(p, 'r', encoding='utf-8') as f:
                        context_text += f"[Table {t_idx+1} HTML]:\n{f.read()}\n\n"

    user_prompt = f"{context_text}\nUSER QUESTION / QUERY:\n{query}\n\nANSWER:"
    has_images = len(visual_elements) > 0
    model_name = "qwen/qwen3.6-27b" if has_images else "openai/gpt-oss-120b"
    llm = ChatGroq(model_name=model_name, temperature=0)

    messages = [SystemMessage(content=system_prompt)]
    if has_images:
        human_content = [{"type": "text", "text": user_prompt}]
        for elem in visual_elements:
            with open(elem["path"], 'rb') as f:
                img_b64 = base64.b64encode(f.read()).decode('utf-8')
            human_content.append({"type": "text", "text": f"[{elem['label']}]:"})
            human_content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}})
        messages.append(HumanMessage(content=human_content))
    else:
        messages.append(HumanMessage(content=user_prompt))

    try:
        response = llm.invoke(messages)
        answer_text = clean_model_output(response.content)
    except Exception as e:
        answer_text = f"Error during generation: {e}"

    if "not have enough information" in answer_text.lower() or "don't have enough information" in answer_text.lower():
        sources = []

    return {"answer": answer_text, "sources": sources}


In [15]:
# =============================================================================
# LangGraph Workflow Assembly & Conditional Routing
# =============================================================================

def decide_to_generate(state: RAGState) -> str:
    """Conditional router: if no docs found and retry < 2, rewrite query; else generate."""
    docs = state.get("filtered_docs", [])
    retries = state.get("retry_count", 0)
    if not docs and retries < 2:
        return "rewrite"
    return "proceed"

# Build the StateGraph
workflow = StateGraph(RAGState)

# Add Nodes
workflow.add_node("transform_query", transform_query_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("rerank", rerank_node)
workflow.add_node("grade_documents", grade_documents_node)
workflow.add_node("prepare_multimodal", prepare_multimodal_context_node)
workflow.add_node("generate", generate_node)

# Connect Edges
workflow.add_edge(START, "transform_query")
workflow.add_edge("transform_query", "retrieve")
workflow.add_edge("retrieve", "rerank")
workflow.add_edge("rerank", "grade_documents")

workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "rewrite": "transform_query",
        "proceed": "prepare_multimodal"
    }
)

workflow.add_edge("prepare_multimodal", "generate")
workflow.add_edge("generate", END)

# Compile the Graph
rag_app = workflow.compile()
print("🚀 LangGraph Multi-Modal RAG Workflow Compiled Successfully!")


# UI Helpers
def render_table_html(raw_html):
    return f"""
    <style>
        .rag-table-wrapper {{ font-family: -apple-system, sans-serif; font-size: 13px; overflow-x: auto; margin: 10px 0; }}
        .rag-table-wrapper table {{ border-collapse: collapse; width: 100%; }}
        .rag-table-wrapper th, .rag-table-wrapper td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
        .rag-table-wrapper th {{ background-color: #f0f0f0; font-weight: 600; }}
        .rag-table-wrapper tr:nth-child(even) {{ background-color: #fafafa; }}
    </style>
    <div class="rag-table-wrapper">{raw_html}</div>
    """


def display_query_result(query, answer, sources):
    print(f"❓ Query: {query}\n")
    print(f"💡 Answer:\n{answer}\n")
    print(f"📚 Sources ({len(sources)}):\n")

    output_area = widgets.Output()

    def make_click_handler(source):
        def handler(b):
            with output_area:
                clear_output(wait=True)
                pdf_label = source.get('source_pdf', '')
                print(f"--- {pdf_label} | chunk {source['chunk_id']} | page {source['page']} | type: {source['type']} ---\n")
                if not source['paths']:
                    print(source['preview'])
                for path in source['paths']:
                    if path.endswith(('.png', '.jpg', '.jpeg')):
                        display(IPImage(filename=path))
                    elif path.endswith('.html'):
                        with open(path, 'r', encoding='utf-8') as f:
                            display(HTML(render_table_html(f.read())))
        return handler

    buttons = [
        widgets.Button(
            description=f"[{i+1}] {s.get('source_pdf', '')[:12]} p.{s['page']} • {s['type']}",
            layout=widgets.Layout(width='auto')
        )
        for i, s in enumerate(sources)
    ]
    for btn, source in zip(buttons, sources):
        btn.on_click(make_click_handler(source))

    if buttons:
        display(widgets.HBox(buttons))
    display(output_area)


🚀 LangGraph Multi-Modal RAG Workflow Compiled Successfully!


In [18]:
query = "What is the functional difference between alpha motor neurons and gamma motor neurons in muscle innervation?"
# Execute via LangGraph
state_input = {"query": query, "retry_count": 0}
graph_result = rag_app.invoke(state_input)

display_query_result(query, graph_result["answer"], graph_result.get("sources", []))


❓ Query: What is the functional difference between alpha motor neurons and gamma motor neurons in muscle innervation?

💡 Answer:
Based on the provided documents, the functional differences between alpha and gamma motor neurons are as follows:

*   **Alpha Motor Neurons:** These are large motor neurons that supply the **extrafusal muscle fibers** (the ordinary muscle fibers responsible for force generation) [Doc 2, p. 4].
*   **Gamma Motor Neurons:** These are smaller motor neurons that supply the **contractile portions of intrafusal fibers** (the specialized fibers within the muscle spindle) [Doc 2, p. 4]. Their specific function is to **regulate the sensitivity of the muscle spindle**, ensuring that this sensitivity is maintained at any given muscle length [Doc 2, p. 4].

📚 Sources (5):



Output()